In [ ]:
# Cell này giúp xuất DataFrame từ TF-IDF artifacts và giải thích giới hạn khôi phục dữ liệu gốc.
from pathlib import Path
import pickle
import pandas as pd
from scipy import sparse

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data" / "processed").exists()),
    None,
 )
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed từ thư mục hiện tại.")

processed_dir = project_root / "data" / "processed"
npz_path = processed_dir / "tfidf_matrix.npz"
pkl_path = processed_dir / "tfidf_matrix.pkl"

if not npz_path.exists() and not pkl_path.exists():
    raise FileNotFoundError("Không tìm thấy cả tfidf_matrix.npz và tfidf_matrix.pkl")

# Ưu tiên đọc từ NPZ; fallback sang PKL
if npz_path.exists():
    tfidf_mat = sparse.load_npz(npz_path)
else:
    with open(pkl_path, "rb") as f:
        tfidf_mat = pickle.load(f)


collab_npz_path = processed_dir / "collab_matrix_normalized.npz"
if collab_npz_path.exists():
    collab_mat = sparse.load_npz(collab_npz_path)
    print(f"\nĐã đọc collaborative matrix từ {collab_npz_path}: shape={collab_mat.shape}, nnz={collab_mat.nnz}")

print(f"\nTF-IDF matrix đã đọc: shape={tfidf_mat.shape}, nnz={tfidf_mat.nnz}")
print(f"Độ thưa của TF-IDF matrix: {(1 - tfidf_mat.nnz / (tfidf_mat.shape[0] * tfidf_mat.shape[1])) * 100:.4f}%")
print(f"\nCollaborative matrix đã đọc: shape={collab_mat.shape}, nnz={collab_mat.nnz}")


print(tfidf_mat[568].data.shape[0])  # In ra một phần nhỏ của ma trận để kiểm tra
print(collab_mat[0].data.shape[0])  # In ra một phần nhỏ của ma trận collaborative để kiểm tra
# # 3) Lưu output
# tfidf_long_out = processed_dir / "tfidf_long_df.parquet"
# tfidf_long_df.to_parquet(tfidf_long_out, index=False)
# print(f"\nĐã lưu long DataFrame: {tfidf_long_out}")

# # (Tuỳ chọn) Nếu muốn file dạng ma trận, có thể rất lớn
# save_sparse_matrix_df = False
# if save_sparse_matrix_df:
#     tfidf_sparse_out = processed_dir / "tfidf_sparse_df.parquet"
#     tfidf_df_sparse.to_parquet(tfidf_sparse_out, index=False)
#     print(f"Đã lưu sparse matrix DataFrame: {tfidf_sparse_out}")

# # 4) Nếu cần 'df gốc', đọc lại từ clean_movies.parquet (nếu có)
# clean_movies_path = processed_dir / "clean_movies.parquet"
# if clean_movies_path.exists():
#     original_df = pd.read_parquet(clean_movies_path)
#     print(f"\nĐọc lại df gốc từ {clean_movies_path}: shape={original_df.shape}")
#     print(original_df.head())
# else:
#     print("\nKhông thấy clean_movies.parquet, nên chưa thể lấy lại df gốc metadata.")
print(collab_mat)


Đã đọc collaborative matrix từ e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\data\processed\collab_matrix_normalized.npz: shape=(255451, 5346), nnz=11412350

TF-IDF matrix đã đọc: shape=(46625, 10000), nnz=739192
Độ thưa của TF-IDF matrix: 99.8415%

Collaborative matrix đã đọc: shape=(255451, 5346), nnz=11412350
20
11


# Split Data

In [11]:
import numpy as np
import scipy.sparse as sparse
from scipy.sparse.linalg import svds

def extract_cf_latent_factors(collab_mat, k_dimensions=50):
    """
    Hàm thực hiện Truncated SVD trên ma trận thưa Collaborative Filtering.
    
    Parameters:
    - collab_mat: Ma trận scipy.sparse (255451 users x 5346 movies)
    - k_dimensions: Số lượng chiều đặc trưng ẩn (Latent factors)
    
    Returns:
    - user_profiles: Ma trận đặc trưng của người dùng (Dense array)
    - item_profiles: Ma trận đặc trưng của video (Dense array)
    """
    print(f"[*] Đang thực hiện phân tích Truncated SVD với k = {k_dimensions}...")
    
    # 1. Thực hiện Singular Value Decomposition trên ma trận thưa
    # Hàm svds sẽ tự động bỏ qua các phần tử 0, tối ưu hóa O(nnz)
    U, sigma, Vt = svds(collab_mat, k=k_dimensions)
    
    # LƯU Ý TOÁN HỌC: 
    # Hàm svds của Scipy trả về các Singular Values theo thứ tự TĂNG DẦN.
    # Ta cần đảo ngược mảng để các thành phần mang nhiều thông tin nhất (variance lớn nhất) lên đầu.
    U = U[:, ::-1]
    sigma = sigma[::-1]
    Vt = Vt[::-1, :]
    
    # 2. Phân bổ trọng số Sigma cho cả User và Item
    # Tạo ma trận đường chéo từ căn bậc hai của sigma
    sigma_sqrt = np.diag(np.sqrt(sigma))
    
    # 3. Tính toán Profile cuối cùng
    # P_u = U * sqrt(Sigma)
    user_profiles = np.dot(U, sigma_sqrt)
    
    # Q_i = V * sqrt(Sigma) (Lưu ý: Vt là V transpose, nên ta cần chuyển vị nó lại thành V)
    item_profiles = np.dot(Vt.T, sigma_sqrt)
    
    print(f"[+] Hoàn thành! Kích thước User Profiles: {user_profiles.shape}")
    print(f"[+] Hoàn thành! Kích thước Item Profiles: {item_profiles.shape}")
    
    return user_profiles, item_profiles

# --- THỰC THI CHƯƠNG TRÌNH ---
# Giả sử em đã load dữ liệu vào biến collab_mat
# collab_mat = sparse.load_npz(collab_npz_path)

# Trích xuất với k = 50 chiều (có thể tinh chỉnh hyperparameter này sau)
user_latent_factors, item_collab_factors = extract_cf_latent_factors(collab_mat, k_dimensions=50)

print(f"Số lượng phần tử khác 0 trong user_latent_factors: {np.count_nonzero(user_latent_factors)}")
print(f"Số lượng phần tử khác 0 trong item_collab_factors: {np.count_nonzero(item_collab_factors)}")
print(f"Tỷ lệ phần tử khác 0 trong user_latent_factors: {(np.count_nonzero(user_latent_factors) / user_latent_factors.size) * 100:.4f}%")
print(f"Tỷ lệ phần tử khác 0 trong item_collab_factors: {(np.count_nonzero(item_collab_factors) / item_collab_factors.size) * 100:.4f}%")
# Lưu lại ra ổ cứng để tái sử dụng ở Pha 3 (tiết kiệm thời gian chạy lại)
# np.save('user_latent_factors_k50.npy', user_latent_factors)
# np.save('item_collab_factors_k50.npy', item_collab_factors)

[*] Đang thực hiện phân tích Truncated SVD với k = 50...
[+] Hoàn thành! Kích thước User Profiles: (255451, 50)
[+] Hoàn thành! Kích thước Item Profiles: (5346, 50)
Số lượng phần tử khác 0 trong user_latent_factors: 12772550
Số lượng phần tử khác 0 trong item_collab_factors: 267300
Tỷ lệ phần tử khác 0 trong user_latent_factors: 100.0000%
Tỷ lệ phần tử khác 0 trong item_collab_factors: 100.0000%


In [10]:
import numpy as np
import scipy.sparse as sparse
from sklearn.decomposition import TruncatedSVD

def extract_cbf_latent_factors(tfidf_mat, k_dimensions=100):
    """
    Thực hiện Latent Semantic Analysis (LSA) trên ma trận thưa TF-IDF.
    
    Parameters:
    - tfidf_mat: Ma trận scipy.sparse (46625 movies x 10000 keywords)
    - k_dimensions: Số lượng chủ đề ẩn (Latent topics) muốn giữ lại.
                    Thường chọn từ 50 đến 300 cho TF-IDF.
    
    Returns:
    - item_content_profiles: Ma trận đặc trưng nội dung của video (Dense array)
    - variance_retained: Tổng lượng thông tin (phương sai) được giữ lại
    """
    print(f"[*] Đang thực hiện Dimensionality Reduction (LSA) với k = {k_dimensions}...")
    
    # 1. Khởi tạo mô hình TruncatedSVD
    # Thuật toán 'randomized' cực kỳ hiệu quả về mặt tính toán cho ma trận thưa lớn
    svd_model = TruncatedSVD(n_components=k_dimensions, algorithm='randomized', random_state=42)
    
    # 2. Fit và Transform ma trận TF-IDF
    # Phép toán này tương đương với việc chiếu X lên không gian V_k: X_reduced = X * V_k = U_k * Sigma_k
    item_content_profiles = svd_model.fit_transform(tfidf_mat)
    
    # 3. Phân tích phương sai (Độ tin cậy của mô hình)
    variance_retained = np.sum(svd_model.explained_variance_ratio_) * 100
    
    print(f"[+] Hoàn thành! Kích thước Item Content Profiles: {item_content_profiles.shape}")
    print(f"[+] Tổng lượng thông tin được giữ lại (Explained Variance): {variance_retained:.2f}%")
    
    # Cảnh báo học thuật: Nếu variance_retained < 60%, em nên cân nhắc tăng k_dimensions.
    
    return item_content_profiles

# --- THỰC THI CHƯƠNG TRÌNH ---
# Giả sử em đã load dữ liệu vào biến tfidf_mat
# tfidf_mat = sparse.load_npz(npz_path)

# Lựa chọn k = 100 (Có thể tùy chỉnh sau khi xem Explained Variance)
item_cbf_factors = extract_cbf_latent_factors(tfidf_mat, k_dimensions=100)
print(f"[+] Số lượng phần tử khác 0 trong ma trận kết quả: {np.count_nonzero(item_cbf_factors)}")  # Kiểm tra số lượng phần tử khác 0 trong ma trận kết quả
print(f"[+] Tỷ lệ phần tử khác 0 trong ma trận kết quả: {(np.count_nonzero(item_cbf_factors) / item_cbf_factors.size) * 100:.4f}%")
# Lưu trữ artifact để phục vụ việc huấn luyện Meta-learner
# np.save('item_content_factors_k100.npy', item_cbf_factors)

[*] Đang thực hiện Dimensionality Reduction (LSA) với k = 100...
[+] Hoàn thành! Kích thước Item Content Profiles: (46625, 100)
[+] Tổng lượng thông tin được giữ lại (Explained Variance): 11.88%
[+] Số lượng phần tử khác 0 trong ma trận kết quả: 4653800
[+] Tỷ lệ phần tử khác 0 trong ma trận kết quả: 99.8134%


In [ ]:
import numpy as np
import sys

def build_hybrid_features(train_user_indices, train_item_cf_indices, 
                          user_latent, item_collab, item_cbf, 
                          cf_to_cbf_map):
    """
    Hàm vector hóa để xây dựng ma trận đặc trưng cho Meta-Learner.
    
    Parameters:
    - train_user_indices: Mảng numpy 1D chứa ID người dùng của tập Train (đã split)
    - train_item_cf_indices: Mảng numpy 1D chứa ID video (hệ CF) của tập Train
    - user_latent: Ma trận user_latent_factors (255451, 50)
    - item_collab: Ma trận item_collab_factors (5346, 50)
    - item_cbf: Ma trận item_cbf_factors (46625, 100)
    - cf_to_cbf_map: Mảng 1D độ dài 5346, dùng để tra cứu: map[cf_id] = cbf_id
    
    Returns:
    - X_train: Ma trận đặc trưng đã ghép nối (N_samples, 200)
    """
    num_samples = len(train_user_indices)
    print(f"[*] Bắt đầu ghép nối đặc trưng cho {num_samples:,} mẫu huấn luyện...")
    
    # BƯỚC 1: DOWNCASTING (Cực kỳ quan trọng để chống OOM)
    # Ép kiểu dữ liệu từ float64 (8 bytes) xuống float32 (4 bytes).
    # Việc này giảm một nửa dung lượng RAM mà hầu như không làm mất độ chính xác của ML.
    user_latent = user_latent.astype(np.float32)
    item_collab = item_collab.astype(np.float32)
    item_cbf = item_cbf.astype(np.float32)
    
    # BƯỚC 2: TÌM INDEX ÁNH XẠ CHO CONTENT-BASED
    # Từ ID của CF, nội suy ra ID tương ứng trong ma trận CBF
    train_item_cbf_indices = cf_to_cbf_map[train_item_cf_indices]
    
    # BƯỚC 3: VECTORIZED LOOKUP (Trích xuất siêu tốc bằng Numpy Advanced Indexing)
    print("[*] Đang trích xuất Latent Factors...")
    X_user = user_latent[train_user_indices]         # Shape: (N_samples, 50)
    X_item_cf = item_collab[train_item_cf_indices]   # Shape: (N_samples, 50)
    X_item_cbf = item_cbf[train_item_cbf_indices]    # Shape: (N_samples, 100)
    
    # BƯỚC 4: FEATURE CONCATENATION
    print("[*] Đang nối các ma trận (Concatenating)...")
    # np.hstack nối các cột lại với nhau theo chiều ngang
    X_train = np.hstack([X_user, X_item_cf, X_item_cbf]) # Shape: (N_samples, 200)
    
    # Tính toán kích thước RAM
    ram_usage_mb = X_train.nbytes / (1024 ** 2)
    print(f"[+] Hoàn thành! Kích thước X_train: {X_train.shape}")
    print(f"[+] Tiêu thụ bộ nhớ của X_train: ~{ram_usage_mb:.2f} MB")
    
    return X_train

# --- THỰC THI CHƯƠNG TRÌNH MẪU ---
# Giả sử em đã có 2 mảng user_indices và item_indices từ tập Train (ví dụ 9 triệu tương tác)
num_rows, num_cols = collab_mat.shape
train_user_idx = np.random.randint(0, num_rows, size=9000000)  # Giả lập 9 triệu user indices
train_item_idx = np.array([...])
cf_to_cbf_mapping_array = np.array([...]) # Mảng ánh xạ do em tự định nghĩa khi làm sạch dữ liệu

X_train = build_hybrid_features(train_user_idx, train_item_idx, 
                                user_latent_factors, item_collab_factors, item_cbf_factors, 
                                cf_to_cbf_mapping_array)

# Đối với nhãn (Target Y), nếu là Explicit Feedback (Rating 1-5):
print(X_train.shape)  # In ra kích thước của ma trận đặc trưng để kiểm tra

[*] Bắt đầu ghép nối đặc trưng cho 1 mẫu huấn luyện...


IndexError: arrays used as indices must be of integer (or boolean) type

# Đánh giá

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

def dcg_at_k(relevances, k):
    """
    Hàm tính Discounted Cumulative Gain (DCG) cho một danh sách.
    Công thức: DCG = sum((2^rel - 1) / log2(rank + 1))
    """
    relevances = np.asarray(relevances)[:k]
    n_items = len(relevances)
    if n_items == 0:
        return 0.0
    
    # Tạo mảng mẫu số: log2(rank + 1). Vị trí index 0 tương ứng rank 1 -> log2(2)
    discounts = np.log2(np.arange(2, n_items + 2))
    
    # Tính DCG
    dcg = np.sum((np.power(2, relevances) - 1) / discounts)
    return dcg

def ndcg_at_k(relevances, k):
    """
    Hàm tính Normalized Discounted Cumulative Gain (NDCG) cho một người dùng.
    """
    # Tính DCG của danh sách hiện tại
    dcg = dcg_at_k(relevances, k)
    
    # Tính Ideal DCG (IDCG) bằng cách sắp xếp danh sách theo độ liên quan giảm dần
    ideal_relevances = np.sort(relevances)[::-1]
    idcg = dcg_at_k(ideal_relevances, k)
    
    # Tránh lỗi chia cho 0
    if idcg == 0:
        return 0.0
    return dcg / idcg

def evaluate_recsys(test_user_ids, y_true, y_pred, k=10):
    """
    Đánh giá toàn diện hệ thống gợi ý với RMSE, MAE và NDCG@K.
    
    Parameters:
    - test_user_ids: Mảng numpy 1D chứa ID người dùng trong tập Test.
    - y_true: Mảng numpy 1D chứa Rating thực tế.
    - y_pred: Mảng numpy 1D chứa Rating dự đoán từ Meta-learner.
    - k: Độ dài danh sách Top-K dùng để tính NDCG.
    
    Returns:
    - Dictionary chứa các chỉ số đánh giá.
    """
    print("[*] Đang tính toán RMSE và MAE toàn cục...")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    print(f"[*] Đang tính toán NDCG@{k} cục bộ cho từng người dùng...")
    # CHIẾN LƯỢC CHỐNG OOM: 
    # Sử dụng Pandas DataFrame để Groupby thay vì khởi tạo lại ma trận thưa
    df_eval = pd.DataFrame({
        'user_id': test_user_ids,
        'y_true': y_true,
        'y_pred': y_pred
    })
    
    # Hàm apply cho từng user: sắp xếp theo y_pred giảm dần và tính NDCG trên y_true
    def user_ndcg(group):
        # Sắp xếp các items của user này theo điểm mà mô hình dự đoán (từ cao xuống thấp)
        sorted_group = group.sort_values('y_pred', ascending=False)
        
        # Lấy mảng độ liên quan thực tế (y_true) theo thứ tự đã sắp xếp
        relevances = sorted_group['y_true'].values
        
        return ndcg_at_k(relevances, k)
    
    # Tính NDCG cho từng user và lấy giá trị trung bình toàn cục
    user_ndcg_scores = df_eval.groupby('user_id').apply(user_ndcg)
    mean_ndcg = user_ndcg_scores.mean()
    
    print("-" * 30)
    print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH")
    print(f"RMSE    : {rmse:.4f}")
    print(f"MAE     : {mae:.4f}")
    print(f"NDCG@{k} : {mean_ndcg:.4f}")
    print("-" * 30)
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        f'NDCG@{k}': mean_ndcg
    }

# --- THỰC THI CHƯƠNG TRÌNH MẪU ---
# Giả sử em đã predict xong từ mô hình XGBoost:
# y_pred = xgb_model.predict(X_test)

# metrics = evaluate_recsys(test_user_indices, y_test, y_pred, k=10)